# 3. Integrações: PostgreSQL, API Go e serviço ML

**Requisito da vaga:** *Implementar integrações entre banco de dados, APIs e plataformas internas*.

Demonstramos as integrações reais do projeto percorrendo o arco de ponta a ponta:

```
Mqtt (simulador) → ingestion (Go) → PostgreSQL → ML service (Python) → API (Go)
```

- **Banco:** leitura direta via `psycopg2` (bronze/operacional).
- **Serviço ML:** HTTP `POST /similar` (contrato JSON, `docs/ml-service.md`).
- **API Go:** HTTP `GET /transformers`, `/{id}/statistics`, `/{id}/telemetry`.

Tudo roda localmente com `make demo`.

> **Pré-requisito:** banco local com dados (`make db && make migrate && make smoke`)
> e o ML service rodando (`make ml-run &`) — ou apenas `make demo`.
>
> Carregar o helper compartilhado (bootstrap de imports, leitores de dados,
> URLs dos serviços) da primeira célula. `notebooks/common.py`.


In [1]:
import sys
sys.path.insert(0, r'/home/carlinhoshk/dev/ETL-Telemetria-Transformadores')
sys.path.insert(0, r'/home/carlinhoshk/dev/ETL-Telemetria-Transformadores/notebooks')
import common
import requests
import pandas as pd
pd.set_option('display.max_columns', None)
print('ML  :', common.ML_URL)
print('API :', common.API_URL)

ML  : http://localhost:8081
API : http://localhost:8080


## 3.1 PostgreSQL → análise

Consulta direta ao modelo operacional (a mesma base que a API serve).

In [2]:
transformers = common.pg_df('''
SELECT transformer_id, rated_power_mva, hv_voltage_kv, application
FROM transformers ORDER BY transformer_id LIMIT 5
''')
transformers

,transformer_id,rated_power_mva,hv_voltage_kv,application
0,TR-001,35.6,115.0,generation
1,TR-002,44.0,230.0,industrial
2,TR-003,30.0,138.0,distribution
3,TR-004,39.8,69.0,distribution
4,TR-005,68.2,34.5,renewable


## 3.2 API Go → contratos HTTP

A API Go (`internal/api`) expõe os dados operacionais. Consumimos os endpoints reais.

In [3]:
import json
r = requests.get(f'{common.API_URL}/transformers/TR-001', timeout=10)
r.raise_for_status()
json.dumps(r.json(), indent=2, ensure_ascii=False)

'{\n  "transformer_id": "TR-001",\n  "rated_power_mva": 35.6,\n  "hv_voltage_kv": 115,\n  "lv_voltage_kv": 11,\n  "frequency_hz": 60,\n  "phase_count": 3,\n  "vector_group": "YNd5",\n  "impedance_percent": 12.5,\n  "cooling_type": "ONAF",\n  "commissioning_year": 2012,\n  "application": "generation",\n  "no_load_loss_kw": 46.9,\n  "load_loss_kw": 370.7,\n  "total_mass_t": 52.8,\n  "length_m": 4.7,\n  "width_m": 3.05,\n  "height_m": 4.14\n}'

In [4]:
r = requests.get(f'{common.API_URL}/transformers/TR-001/statistics', timeout=10)
r.raise_for_status()
json.dumps(r.json(), indent=2)

'{\n  "transformer_id": "TR-001",\n  "count": 6,\n  "min_load_percent": 53.3,\n  "max_load_percent": 66.6,\n  "avg_load_percent": 56.56666666666667,\n  "max_oil_temperature_c": 55,\n  "max_winding_temperature_c": 70,\n  "avg_winding_temperature_c": 29.266666666666666,\n  "avg_thermal_stress_index": 0.249\n}'

## 3.3 Python ML → serviço de IA

O mecanismo de similaridade roda no **serviço Python ML** (independência de plataforma): a API Go delega para ele. Consumimos a mesma rota.

In [5]:
matches = common.similar_for('TR-001', top_k=5)
matches

,transformer_id,score
0,TR-018,0.5982
1,TR-039,0.5823
2,TR-033,0.5452
3,TR-037,0.4963
4,TR-003,0.4736


In [6]:
# Prova do fluxo completo: ID alvo -> API Go -> ML service -> top-k com score.
target = 'TR-001'
api = requests.get(f'{common.API_URL}/transformers/{target}', timeout=10).json()
ml  = requests.post(f'{common.ML_URL}/similar', timeout=15, json={
        'target': api,
        'candidates': common.to_plain_records(
            common.pg_df('''SELECT * FROM transformers
                        WHERE transformer_id <> %s''', (target,))),
        'top_k': 5}).json()
pd.DataFrame(ml['results'])

,transformer_id,score
0,TR-018,0.5982
1,TR-039,0.5823
2,TR-033,0.5452
3,TR-037,0.4963
4,TR-003,0.4736


## Conclusão

- Integrações banco ↔ API ↔ serviço ML funcionam de ponta a ponta e consumidas via contratos JSON estáveis.
- Mesma chave (`transformer_id`) atravessa todas as camadas — sem conversão ad-hoc.